# House Recognition Challenge - SOTA Pipeline



### เทคนิคหลักที่ใช้:
1. **Transfer Learning**: ใช้โมเดล **EfficientNet** ที่ถูกเทรนมาแล้วจาก ImageNet มาปรับจูน (Fine-tune)
2. **Data Augmentation**: ใช้ไลบรารี `albumentations` เพื่อจำลองภาพในมุมมองต่างๆ ป้องกันการ Overfitting
3. **5-Fold Stratified Cross Validation**: แบ่งข้อมูลเป็น 5 ส่วน เพื่อเทรนโมเดล 5 ตัว ช่วยให้ผลลัพธ์เสถียรและแม่นยำขึ้น
4. **Test-Time Augmentation (TTA)**: ในตอนทำนายผล จะมีการพลิกภาพซ้าย-ขวาแล้วทำนายซ้ำอีกครั้งเพื่อความแม่นยำสูงสุด

In [ ]:
# ติดตั้งไลบรารีที่จำเป็น (ถ้ายังไม่มี)
!pip install timm albumentations scikit-learn pandas opencv-python matplotlib tqdm -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2                          # สำหรับอ่านและจัดการรูปภาพ
import matplotlib.pyplot as plt      # สำหรับแสดงกราฟและภาพ
from tqdm.notebook import tqdm       # แถบแสดงความก้าวหน้า (Progress Bar)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm                          # คลังโมเดล Computer Vision ระดับโลก
import albumentations as A            # ไลบรารีทำ Data Augmentation ที่เร็วที่สุด
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ฟังก์ชันสำหรับล็อกค่า Random เพื่อให้รันกี่ครั้งผลก็ได้แบบเดิม (Reproducibility)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(42)

# ตรวจสอบว่ามี GPU ให้ใช้ไหม (CUDA สำหรับ Nvidia, MPS สำหรับ Apple Silicon, หรือ CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Using device:', device)

## 1. การตั้งค่าไฮเปอร์พารามิเตอร์ (Configuration)
เราสามารถปรับเปลี่ยนค่าตรงนี้ได้เพื่อขยับคะแนนให้สูงขึ้น

In [ ]:
class CFG:
    model_name = 'tf_efficientnet_b2.ns_jft_in1k' # ชื่อโมเดลเริ่มต้น (EfficientNet-B2)
    size = 260                  # ขนาดภาพ (กว้างxยาว) ที่จะป้อนเข้าโมเดล
    epochs = 7                  # จำนวนรอบในการเทรนต่อ 1 Fold
    batch_size = 32             # จำนวนภาพที่ส่งเข้าโมเดลพร้อมกัน (ถ้า RAM เต็มให้ลดลง)
    lr = 1e-4                   # อัตราการเรียนรู้ (Learning Rate)
    weight_decay = 1e-6         # การปรับแต่งเพื่อลด Overfitting
    n_fold = 5                  # จำนวนการแบ่งข้อมูล (K-Fold)
    trn_fold = [0, 1, 2, 3, 4]  # Fold ที่เราจะทำการเทรนจริง (รันทุก Fold เพื่อความแม่นยำ)
    
    # ที่อยู่ของข้อมูลต่างๆ
    train_dir = 'data/train/train'
    test_dir = 'data/test/test'
    train_csv = 'data/train.csv'
    sub_csv = 'data/sample_submission.csv'

## 2. การเตรียมข้อมูลและแบ่งกลุ่ม (K-Fold Split)
เทคนิค **StratifiedKFold** ช่วยให้แต่ละกลุ่มมีสัดส่วนของคลาส (0 และ 1) เท่าๆ กับข้อมูลจริง

In [ ]:
train_df = pd.read_csv(CFG.train_csv)
train_df['file_path'] = train_df['image_name'].apply(lambda x: os.path.join(CFG.train_dir, x))

skf = StratifiedKFold(n_splits=CFG.n_fold, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['class'])):
    train_df.loc[val_idx, 'fold'] = fold
train_df['fold'] = train_df['fold'].astype(int)

print(f"ข้อมูลทั้งหมด: {len(train_df)} รูป")
train_df.head()

## 3. Dataset & Albumentations (หัวใจของการทำคะแนน)
ตรงนี้คือการสอนให้ AI รู้จักภาพในหลายมิติ เช่น พลิกภาพ หรือขยับตำแหน่ง เพื่อไม่ให้ AI จำภาพเดิมได้

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df
        self.file_paths = df['file_path'].values
        self.transform = transform
        self.is_test = is_test
        if not self.is_test:
            self.labels = df['class'].values
            
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        image = cv2.imread(file_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
            
        if self.is_test:
            return image
        else:
            label = torch.tensor(self.labels[idx], dtype=torch.float)
            return image, label

def get_transforms(data):
    if data == 'train':
        return A.Compose([
            # สุ่มตัดและขยายภาพ ให้ AI รู้จักวัตถุในระยะที่ต่างกัน
            A.RandomResizedCrop(size=(CFG.size, CFG.size), scale=(0.8, 1.0)),
            # พลิกภาพซ้าย-ขวา
            A.HorizontalFlip(p=0.5),
            # สุ่มขยับ, ย่อ/ขยาย, และหมุนภาพเล็กน้อย
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
            # ปรับแสง สี ความเข้ม ให้ภาพดูสว่างหรือมืดต่างกัน
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
            # ปรับค่าสีให้เป็นมาตรฐานกลางเพื่อให้โมเดลเรียนรู้ได้ง่ายขึ้น
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    elif data == 'valid':
        return A.Compose([
            # สำหรับตอนทดสอบ เราแค่เปลี่ยนขนาดภาพให้ถูกต้อง ไม่สุ่มพลิก
            A.Resize(height=CFG.size, width=CFG.size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

## 4. นิยามโครงสร้างโมเดล (Model Architecture)
เราดึงโครงสร้างที่เก่งๆ (EfficientNet) มา แล้วเปลี่ยนส่วนท้าย (Classifier) ให้ผลลัพธ์เหลือเพียง 1 ค่า (ความน่าจะเป็น)

In [ ]:
class HouseModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        # ดึงโมเดลมาตรฐานมาโดยตัดหัวเดิม (Classifier) ทิ้ง
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0) 
        in_features = self.model.num_features
        # สร้างส่วนหัวใหม่เพื่อทำนาย Binary Classification (0 หรือ 1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),              # สุ่มปิด Node บางตัวเพื่อลด Overfitting
            nn.Linear(in_features, 1)    # ปลายทางคือ 1 Output
        )
        
    def forward(self, x):
        features = self.model(x)
        output = self.classifier(features)
        return output

## 5. ฟังก์ชันการเทรน (Training Loop)
ใช้ **BCEWithLogitsLoss** ซึ่งเหมาะมากกับโจทย์แยกแยะ 2 คลาส

In [ ]:
def train_one_fold(fold):
    print(f"\n========== เริ่มเทรน Fold {fold} ==========")
    trn_idx = train_df[train_df['fold'] != fold].index
    val_idx = train_df[train_df['fold'] == fold].index
    
    train_folds = train_df.loc[trn_idx].reset_index(drop=True)
    valid_folds = train_df.loc[val_idx].reset_index(drop=True)
    
    train_dataset = HouseDataset(train_folds, transform=get_transforms('train'))
    valid_dataset = HouseDataset(valid_folds, transform=get_transforms('valid'))
    
    # num_workers=0 เพื่อป้องกัน Error ตอนโหลดภาพข้ามโปรเซส
    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CFG.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)
    
    model = HouseModel(CFG.model_name, pretrained=True)
    model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    # Scheduler จะช่วยลดอัตราการเรียนรู้ลงเรื่อยๆ เมื่อเทรนไปนานๆ เพื่อให้คำตอบแม่นยำที่สุด
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=1e-6)
    
    best_score = 0.
    
    for epoch in range(CFG.epochs):
        model.train()
        train_loss = 0
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG.epochs} [Train]', leave=False):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            y_preds = model(images)
            loss = criterion(y_preds.view(-1), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        scheduler.step()
        
        model.eval()
        valid_loss = 0
        preds = []
        targets = []
        with torch.no_grad():
            for images, labels in tqdm(valid_loader, desc=f'Epoch {epoch+1}/{CFG.epochs} [Valid]', leave=False):
                images, labels = images.to(device), labels.to(device)
                y_preds = model(images)
                loss = criterion(y_preds.view(-1), labels)
                valid_loss += loss.item()
                preds.append(y_preds.sigmoid().cpu().numpy())
                targets.append(labels.cpu().numpy())
                
        preds = np.concatenate(preds)
        targets = np.concatenate(targets)
        val_acc = accuracy_score(targets, (preds > 0.5).astype(int))
        
        print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_score:
            best_score = val_acc
            # บันทึกโมเดลที่ดีที่สุดของแต่ละกลุ่มข้อมูลไว้
            torch.save(model.state_dict(), f'{CFG.model_name}_fold{fold}_best.pth')
            
    print(f"จบ Fold {fold} - Accuracy ที่ดีที่สุด: {best_score:.4f}")

## 6. สั่งให้เริ่มเทรนข้อมูล (Let's Go!)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# รันการเทรนตามจำนวน Fold ที่เลือกไว้
for fold in CFG.trn_fold:
    train_one_fold(fold)

## 7. รวมพลังโมเดลและทำนายผลจริง (Ensemble & TTA)
ขั้นตอนนี้คือการนำโมเดลทั้ง 5 ตัวที่รันเสร็จแล้วมาช่วยกันโหวตคำตอบ เพื่อให้ได้คะแนนสูงสุด

In [ ]:
def inference():
    sub_df = pd.read_csv(CFG.sub_csv)
    sub_df['file_path'] = sub_df['id'].apply(lambda x: os.path.join(CFG.test_dir, str(x) + '.jpg'))
    
    test_dataset = HouseDataset(sub_df, transform=get_transforms('valid'), is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size * 2, shuffle=False, num_workers=0)
    
    predictions = np.zeros(len(sub_df))
    
    # เรียกโมเดลทั้ง 5 ร่างกลับมาเข้าประจำการ
    models = []
    for fold in CFG.trn_fold:
        model = HouseModel(CFG.model_name, pretrained=False)
        model.load_state_dict(torch.load(f'{CFG.model_name}_fold{fold}_best.pth', map_location=device))
        model.to(device)
        model.eval()
        models.append(model)
        
    print("กำลังทำนายผลด้วยเทคนิค Ensemble และ TTA...")
    with torch.no_grad():
        for i, images in enumerate(tqdm(test_loader)):
            images = images.to(device)
            
            # 1. ทำนายด้วยภาพปกติ
            batch_preds = np.zeros((images.size(0),))
            for model in models:
                out = model(images).sigmoid().view(-1).cpu().numpy()
                batch_preds += out / len(models)
            
            # 2. ทำนายด้วยภาพพลิกซ้ายขวา (TTA)
            images_flipped = torch.flip(images, dims=[3]) 
            batch_preds_flipped = np.zeros((images.size(0),))
            for model in models:
                out = model(images_flipped).sigmoid().view(-1).cpu().numpy()
                batch_preds_flipped += out / len(models)
                
            # เฉลี่ยคะแนนจากโมเดล 5 ตัว และภาพ 2 รูปแบบ -> ได้ผลลัพธ์ที่เสถียรที่สุด
            final_pred = (batch_preds + batch_preds_flipped) / 2
            
            start_idx = i * CFG.batch_size * 2
            end_idx = start_idx + images.size(0)
            predictions[start_idx:end_idx] = final_pred
            
    sub_df['answer'] = (predictions > 0.5).astype(int)
    sub_df[['id', 'answer']].to_csv('submission_optimized.csv', index=False)
    print("บันทึกไฟล์เรียบร้อยที่ submission_optimized.csv!")
    return sub_df

ans_df = inference()
ans_df[['id', 'answer']].head()